# PRÁCTICA 2: laboratorio de árboles de decisión
##### Alumnos
- Alexandre Lorenzo Martínez
- Lucía Pérez González
- Manuel Ramallo Blanco

Na presente práctica (...)

### 2.1- Construcción dunha árbore de decisión
(...)

In [1]:
import pandas as pd
import numpy as np

class Node:

    def __init__(self, derecho=None, izquierdo=None, atributo=None, umbral=None, valor=None):
        self.derecho = derecho
        self.izquierdo = izquierdo
        self.atributo = atributo
        self.umbral = umbral
        self.valor = valor

    def _is_leaf(self):
        return self.valor is not None


class DecisionTree:

    def  __init__(self, max_depth=None , min_samples_split=2, max_leaf_nodes=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_leaf_nodes = max_leaf_nodes
        self.n_leaf_nodes = 0

    def _is_finished(self, profundidad):
        # Compróbase si se chegou a profundidade máxima
        if self.max_depth is not None and profundidad >= self.max_depth:
            return True
        
        # Compróbase se se chegou ao número máximo de nodos folla
        if self.max_leaf_nodes is not None and self.n_leaf_nodes >= self.max_leaf_nodes:
            return True
        
        return False
    
    def _most_common_label(self, y):
        if len(y) == 0:
            return None
        return np.bincount(y).argmax()
    
    def _build_tree(self, X, y, atributos_disponibles, profundidad=0):
        if self._is_finished(profundidad) or len(y) < self.min_samples_split:
            self.n_leaf_nodes += 1
            return Node(valor=self._most_common_label(y))
        
        atributo, umbral = self._best_split(X, y, atributos_disponibles)
        if atributo is None:
            self.n_leaf_nodes += 1
            return Node(valor=self._most_common_label(y))

        X_izq, y_izq, X_der, y_der = self._create_split(X, y, atributo, umbral)
        if len(y_izq) == 0 or len(y_der) == 0:
            self.n_leaf_nodes += 1
            return Node(valor=self._most_common_label(y))

        nuevos_atributos = atributos_disponibles.copy()
        nuevos_atributos.remove(atributo)

        nodo_izquierdo = self._build_tree(X_izq, y_izq, nuevos_atributos, profundidad + 1)
        nodo_derecho = self._build_tree(X_der, y_der, nuevos_atributos, profundidad + 1)

        return Node(
            izquierdo=nodo_izquierdo,
            derecho=nodo_derecho,
            atributo=atributo,
            umbral=umbral
        )

    def _entropy(self, labels):
        if len(labels) == 0:
            return 0

        valores, counts = np.unique(labels, return_counts=True)
        proporciones = counts / len(labels)

        return -np.sum(proporciones * np.log2(proporciones))

    def _information_gain(self, X, y, atrib, entropia_anterior):
        columna = X[:, atrib]
        valores_unicos = np.unique(columna)
        #imprimir columna y valores únicos para debug
        if len(valores_unicos) == 2:
            umbral = np.mean(valores_unicos)
        else:
            umbral = np.median(columna)
            
        mask_izq = columna <= umbral
        mask_der = columna > umbral
        
        y_izq = y[mask_izq]
        y_der = y[mask_der]
            
        e_izq = self._entropy(y_izq)
        e_der = self._entropy(y_der)
        
        n_total = len(y)
        n_izq = len(y_izq)
        n_der = len(y_der)
        
        entropia_final = (n_izq / n_total) * e_izq + (n_der / n_total) * e_der
        ganancia = entropia_anterior - entropia_final

        return ganancia, umbral

    def _create_split(self, X, y, atributo, umbral):
            # Seleccionase a columna do atributo a dividir
            columna = X[:, atributo]

            # Créanse as máscaras booleanas (arrays booleanos)
            mask_izq = columna <= umbral
            mask_der = columna > umbral

            # Aplícanse as máscaras para obter os subconxuntos de datos
            X_izq = X[mask_izq]
            y_izq = y[mask_izq]

            X_der = X[mask_der]
            y_der = y[mask_der]

            return X_izq, y_izq, X_der, y_der

    def _best_split(self, X, y, atributos_disponibles):

        mejor_ganancia = -1
        mejor_atributo = None
        mejor_umbral = None

        entropia_anterior = self._entropy(y)

        for atrib in atributos_disponibles:
            ganancia, umbral = self._information_gain(X, y, atrib, entropia_anterior)

            if ganancia > mejor_ganancia:
                mejor_ganancia = ganancia
                mejor_atributo = atrib
                mejor_umbral = umbral
        
        if mejor_ganancia <= 0:
            return None, None

        return mejor_atributo, mejor_umbral

    def _traverse_tree(self, muestra):
        #Se supone que se guarda la raíz como atributo del árbol
        nodo = self.raiz

        while not nodo._is_leaf():
            if muestra[nodo.atributo] <= nodo.umbral:
                nodo = nodo.izquierdo
            else:
                nodo = nodo.derecho

        return nodo.valor

    def fit(self, X, y):
        self.n_features = X.shape[1]
        atributos = list(range(self.n_features))
        self.raiz = self._build_tree(X, y, atributos.copy())

    def predict(self, X):
        #Recorre cada muestra de X y devuelve el conjunto de predicciones
        predictions = []
        for muestra in X:
            prediction = self._traverse_tree(muestra)
            predictions.append(prediction)
        return np.array(predictions)
    
    def print_arbol(self, nodo=None, nivel=0):
        features = ["age_young", "age_pre-presbyopic", "age_presbyopic", "prescription_myope", "prescription_hypermetrope", "astigmatic_no", "astigmatic_yes", "tear_rate_reduced", "tear_rate_normal"]
        label_values = ["no lenses", "soft", "hard"]
        if nodo is None:
            nodo = self.raiz
        
        if nodo._is_leaf():
            print('|   ' * nivel + '-> Valor:', label_values[nodo.valor])
        else:
            print('|   ' * nivel + f'-> Atributo: {features[nodo.atributo]} <= {nodo.umbral}')
            self.print_arbol(nodo.izquierdo, nivel + 1)
            self.print_arbol(nodo.derecho, nivel + 1)

## 2.2- Adestramento e validación da árbore
(...)

## 2.3- Comparación cun modelo de caixa negra: modelo neuronal (MLP)
(...)